# 03 · Nested spatial cross-validation (the long run)

**How it works (same design as V3)**

```
for each outer fold k = 1..5 (blockCV folds):
    test  = boreholes in fold k
    train = all other boreholes (with a purge: minus those closer than the purge distance)
    inner CV = the other 4 outer folds
        Optuna tries hyperparameter sets (50 for RF/XGB/MLP, 30 for GWRF/GWXGB) and the
        weight of filled cells; each set is scored by the inner Macro-F1 at its best threshold
    refit the best set on all training boreholes -> predict fold k
```

**Runs** (80 outer folds in total)

| Block size | Purge | Models |
|---|---|---|
| 1× (main) | none | RF, XGB, MLP, GWRF, GWXGB + depth-only |
| 2× and 3× | none | RF, XGB + depth-only |
| 1× | 1× and 2× the block size | RF, XGB |

Inputs: Easting, Northing, top elevation of the cell. Test cells: measured + filled (scores are also given for measured cells only, in notebook 04).

**Run time: about 7 hours on a fast 4-core PC; plan 0.5-1.5 days on the school PC.** Every outer fold is saved when it finishes. If the PC restarts, just run the notebook again: finished folds are loaded, and it continues with the next one. To get the main results sooner, set `RUN_ONLY = ["x1_purge0"]` first, then `None` for the rest.

In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import helpers as H

S = H.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| area:", S["area"]["area"])
import joblib

RUN_ONLY = None        # None = all runs; or parts of run names, e.g. ["x1_purge0"] or ["XGB_x1"]

cells = pd.read_csv(H.processed_dir(S) / "cells.csv")
folds = {m: H.load_folds(S, m) for m in S["blockcv"]["multiples"]}
block_m = folds[1][1]
runs = H.run_list(S, block_m)
if RUN_ONLY:
    runs = [r for r in runs if any(s in r["tag"] for s in RUN_ONLY)]
outer = S["quick"]["outer_folds"] if S["run"]["quick"] else list(range(1, S["blockcv"]["k"] + 1))
PRIV = H.out_dir(S, "03_spatial_cv", private=True)
(PRIV / "checkpoints").mkdir(exist_ok=True); (PRIV / "models").mkdir(exist_ok=True)
print(f"Block size {block_m:.1f} m | {len(runs)} runs x {len(outer)} outer folds | {len(cells)} cells, "
      f"{cells.BH_Code.nunique()} boreholes")
display(pd.DataFrame(runs))

## Run (or continue)

In [ ]:
for r in runs:
    fold_ids, block = folds[r["multiple"]]
    purge_m = r["purge_x"] * block_m
    key = H.settings_key(S, cells, fold_ids) + f"_{purge_m:.3f}"
    for k in outer:
        ck = PRIV / "checkpoints" / f"{r['tag']}_fold{k}.joblib"
        if ck.exists():
            if joblib.load(ck)["key"] != key:
                raise RuntimeError(f"{ck.name} was made with other data or settings: delete the folder "
                                   f"{PRIV / 'checkpoints'} (and models) to start again.")
            continue
        res = H.run_fold(r["model"], cells, fold_ids, k, purge_m, S, r["tag"])
        res["predictions"] = res["predictions"].assign(model=r["model"], multiple=r["multiple"], block_m=block,
                                                       purge_x=r["purge_x"], purge_m=purge_m)
        res["info"].update(model=r["model"], multiple=r["multiple"], block_m=block, purge_m=purge_m)
        if r["multiple"] == 1 and r["purge_x"] == 0:       # keep the fold models of the main runs for notebook 06
            joblib.dump(res["model"], PRIV / "models" / f"{r['tag']}_fold{k}.joblib", compress=3)
        res.pop("model")
        joblib.dump({**res, "key": key}, ck, compress=3)
        i = res["info"]
        print(f"{time.strftime('%H:%M')}  {r['tag']:<24} fold {k}: inner Macro-F1 {i['inner_macro_f1']:.3f}, "
              f"threshold {i['threshold']:.2f}, {i['seconds'] / 60:.1f} min", flush=True)
print(f"All requested folds done ({(time.time() - T0) / 3600:.2f} h in this session).")

## Collect the results

In [ ]:
preds, infos, trials = [], [], []
for f in sorted((PRIV / "checkpoints").glob("*.joblib")):
    res = joblib.load(f)
    preds.append(res["predictions"]); infos.append(res["info"])
    trials.append(res["trials"].assign(tag=res["info"]["tag"], outer_fold=res["info"]["outer_fold"]))
preds = pd.concat(preds, ignore_index=True)
info = pd.DataFrame(infos)
preds.to_csv(PRIV / "outer_predictions.csv", index=False)
pd.concat(trials, ignore_index=True).to_csv(PRIV / "optuna_trials.csv", index=False)
OUT = H.out_dir(S, "03_spatial_cv")
info.to_csv(OUT / "fold_info.csv", index=False)
done = info.groupby("tag").agg(folds=("outer_fold", "count"), hours=("seconds", lambda s: s.sum() / 3600))
display(done.round(2))
print(f"{len(preds)} test predictions saved. Next: notebook 04.")